# 07 — The Specialist Roster

Each exploration node runs a team of 6 core specialists in sequence:

**Surveyor → DataDigger → Theorist → Analyst → Innovator → Critic**

Plus an optional **Connector** (khive-only) that weaves discoveries into the knowledge graph.

Each specialist has specific tools and depth-aware behavior.

In [ ]:
from lionag2.research.prompts import (
    CONNECTOR,
    build_node_instruction,
    build_roster,
)

## The roster

Each specialist is a dict with `name`, `role`, `tools`, and `prompt`.

In [ ]:
# build_roster adapts prompts based on available tools
roster = build_roster(has_khive=False, has_exa=True)

for spec in roster:
    print(f"\n{'='*60}")
    print(f"{spec['name'].upper()} — {spec['role']}")
    print(f"Tools: {spec['tools']}")
    print(f"Prompt: {spec['prompt'][:150]}...")

print(f"\n{'='*60}")
print(f"{CONNECTOR['name'].upper()} — {CONNECTOR['role']}")
print(f"Tools: {CONNECTOR['tools']}")
print("(Only added when KHIVE_API_KEY is set)")

## Tool mapping

| Specialist | search | fetch | run_code | memory | graph | messages |
|---|---|---|---|---|---|---|
| Surveyor | + | + | | + | + | + |
| DataDigger | + | + | | + | + | |
| Theorist | + | + | | + | | |
| Analyst | + | + | + | + | + | |
| Innovator | + | + | | + | + | |
| Critic | + | + | | + | + | + |
| Connector | | | | + | + | |

All specialists also get the 4 emission tools (`emit_finding`, `request_depth`, `emit_contradiction`, `emit_pivot`).

Tool tags map to real toolkits in `_resolve_tools()` — each agent gets a **fresh** toolkit instance to avoid deepcopy issues.

## Depth-aware prompting

The instruction changes with depth — root maps the territory, depth 1 drills mechanism, depth 2+ chases empirical specifics.

In [ ]:
for depth in range(3):
    instruction = build_node_instruction(
        "What causes high-Tc superconductivity?",
        depth=depth,
        max_depth=3,
    )
    print(f"\n--- Depth {depth} ---")
    print(instruction[:300])
    print("...")

## Team runner

Agents within a team coordinate via **handoff**: each agent does its work, then calls `handoff("next_agent")` to route to the most relevant specialist, or `handoff("done")` to end. Default fallback is roster order.

```python
# Handoff-based team loop
current_idx = 0
for turn in range(max_turns):
    agent = agents[current_idx]
    reply = await agent.ask(prompt, stream=agent_stream)
    # Observer on HandoffRequested routes to next agent
    if handoff_target == "done":
        break
    current_idx = roster_index[handoff_target]
```

Each agent gets its own named stream. Bridge observers forward typed events (FindingEmitted, DepthRequested, etc.) to the engine for reactive depth expansion.

## URL capture observer

Every agent has a `ToolResultsEvent` observer that captures `title → url` mappings from Exa search results. Observers can be **sync or async** — this one is synchronous since it only writes to a dict:

```python
@agent.observer(ToolResultsEvent)
def _capture_urls(event: ToolResultsEvent) -> None:
    for r in event.results:
        for hit in getattr(data, 'results', []):
            self.title_to_url[hit.title] = hit.url
```

This dict grounds citations at render time — titles are verified against what the search actually returned.

## Up next

When khive is available, lionag2 uses it as the persistent knowledge backend. Tutorial 08 shows the `KhiveKnowledgeStore` and `KhiveToolkit`.